#Универсальный Конвейер Обработки ИК-Спектров (JCAMP-DX Parser Pipeline)

Реализация воспроизводимого пайплайна для пакетной загрузки, стандартизации, интерполяции ИК-спектров из формата `JCAMP-DX` (базы данных NIST, SDBS и др.), восстановления отсутствующих химических метаданных и инкрементального расширения единого датасета.

## Основные возможности
* **Интеграция и нормализация спектров:** Быстрый парсинг файлов .jdx / .dx с помощью jcamp, нормализация названий колонок по словарю COLUMN_MAPPING и векторизованная интерполяция всех спектров на единую сетку волновых чисел (500–4100 cm⁻¹, шаг 2.0 cm⁻¹) с использованием numpy.interp в формате float32
* **Каскадное обогащение химических данных:** Параллельный поиск пропущенных SMILES (ThreadPoolExecutor) по логике fetch_smiles_smart — приоритетно по CAS через PubChem API и NCI Cactus API, с резервным поиском по названию соединения (Title).
* **Расчет молекулярных свойств через RDKit:** Автоматическое вычисление точной молярной массы (Mw) и брутто-формулы (Formula) на основе валентных структур с помощью инструментов RDKit (ExactMolWt, CalcMolFormula) вместо внешних библиотек элементов.
*  **Взвешенная дедупликация данных (calc_score):** Расчет индекса информативности info_score на основе весов заполненных полей для автоматического отбора наиболее полной записи при удалении дубликатов по CAS.
* **Разметка 17 функциональных групп:** Формирование вектора бинарных меток $y \in \{0, 1\}^{17}$ на основе оптимизированного подструктурного поиска RDKit SMARTS
* **Аналитика и визуализация датасета:** Проверка пропусков и качества валидации данных (аудит InChI), а также визуализация распределений классов и мульти-меток на базе seaborn.


### 1. Установка зависимостей и импорт модулей

In [ ]:
# 1. Системные и встроенные модули Python
import os
import io
import gc
import re
import sys
import glob
import contextlib
import requests
from pathlib import Path
from IPython.display import display
from typing import List, Optional, Tuple, Union
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed

# 2. Базовые вычислительные ядра C/C++ 
import numpy as np
import scipy
import matplotlib.pyplot as plt
import seaborn as sns

# 3. Тяжелые хемоинформационные и C++ обертки (RDKit)
from rdkit import Chem, RDLogger, rdBase
from rdkit.Chem import Descriptors, inchi, rdMolDescriptors, Draw
RDLogger.DisableLog('rdApp.*')# Отключение предупреждений RDKit
import jcamp
jcamp_readfile = getattr(jcamp, 'jcamp_readfile', 
                 getattr(jcamp, 'jcamp_reader', 
                 getattr(jcamp, 'JCAMP_reader', 
                 getattr(jcamp, 'readfile', 
                 getattr(jcamp, 'read', None)))))

if jcamp_readfile is None:
    raise ImportError("Не удалось найти подходящую функцию чтения в установленном пакете 'jcamp'.")

# 4. Высокоуровневая обработка данных и утилиты
import pandas as pd
from tqdm import tqdm

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/IR')
else:
    BASE_DIR = Path.cwd()

DB_PATH = str(BASE_DIR / "db" / "processed_dataset2807.pkl")
NEW_DATA_FOLDER = str(BASE_DIR / "data" / "new_batch")
CSV_PATH = str(BASE_DIR / "compounds.csv")

print(f"Среда: {'Google Colab' if IN_COLAB else 'Локальный ПК'}")
print(f"База данных: {DB_PATH}")

### Вспомогательные функции 
(интерполяция, нормализация CAS, маска невалидных ячеек, удаление дублированных колонок)

#### Интерполяция

In [ ]:
def universal_interpolate(x, y, borders=[500, 4100], deltax=2):
    """Линейная интерполяция спектра на единую сетку волновых чисел."""
    x_grid = np.arange(borders[0], borders[1] + deltax, deltax, dtype=np.float32)
    y_interp = np.interp(x_grid, x, y, left=0.0, right=0.0).astype(np.float32)
    return x_grid, y_interp

#### Нормализация CAS

In [ ]:
def normalize_cas(cas):
    if pd.isna(cas) or cas is None:
        return None
    cas_str = str(cas).strip()
    if cas_str.endswith('.0'):
        cas_str = cas_str[:-2]
    if re.match(r'^\d{2,7}-\d{2}-\d$', cas_str):
        return cas_str
    clean_digits = re.sub(r'\D', '', cas_str)
    if len(clean_digits) >= 5:
        return f"{clean_digits[:-3]}-{clean_digits[-3:-1]}-{clean_digits[-1]}"
    return cas_str if clean_digits else None

### Спектральная интеграция и чтение файлов

In [ ]:
def process_spectrum_folder(folder_path: str, borders: List[int] = [500, 4100], deltax: int = 2, save_path: Optional[str] = None) -> pd.DataFrame:
    """Пакетный парсинг JCAMP-DX файлов с обогащением хемоинформатикой RDKit.""" 
    # Сбор файлов
    if '*' in folder_path or '?' in folder_path:
        files = glob.glob(folder_path)
    else:
        files = glob.glob(os.path.join(folder_path, '*'))

    if not files:
        print(f'Файлы по пути "{folder_path}" не найдены.')
        return pd.DataFrame()

    dataset_list = []
    empty_files_counter = 0
    error_examples = []

    print(f'Считывание и обработка файлов ({len(files)} шт.)...')

    # Последовательный парсинг
    for file in tqdm(files, desc="Парсинг спектров"):
        try:
            if os.path.getsize(file) == 0:
                empty_files_counter += 1
                continue

            with open(file, 'r', encoding='utf-8', errors='ignore') as f:
                lines = f.readlines()

            if not lines:
                empty_files_counter += 1
                continue

            clean_lines = []
            has_multiple_blocks = False
            for line in lines:
                clean_lines.append(line)
                if line.strip().startswith('##END='):
                    has_multiple_blocks = True
                    break

            # Чтение JCAMP
            log_capture = io.StringIO()
            with contextlib.redirect_stdout(log_capture), contextlib.redirect_stderr(log_capture):
                if has_multiple_blocks and len(clean_lines) < len(lines):
                    virtual_file = io.StringIO(''.join(clean_lines))
                    sample = jcamp_readfile(virtual_file)
                else:
                    sample = jcamp_readfile(file)

            if not isinstance(sample, dict) or 'x' not in sample or 'y' not in sample:
                empty_files_counter += 1
                continue

            y_arr = np.array(sample['y'], dtype=np.float32) if sample['y'] is not None else np.array([], dtype=np.float32)
            if len(y_arr) == 0:
                empty_files_counter += 1
                continue

            x_arr = np.array(sample['x'], dtype=np.float32) if sample['x'] is not None else np.array([], dtype=np.float32)

            # Выравнивание размерностей осей
            if len(x_arr) != len(y_arr):
                first_x = float(sample.get('firstx', x_arr[0] if len(x_arr) > 0 else 4000.0))
                last_x = float(sample.get('lastx', x_arr[-1] if len(x_arr) > 0 else 400.0))
                x_arr = np.linspace(first_x, last_x, len(y_arr), dtype=np.float32)

            sample['x'] = x_arr.astype(np.float32)
            sample['y'] = y_arr.astype(np.float32)

            # Нормализация Y (Absorbance)
            if 'yunits' in sample and str(sample['yunits']).upper() in ['TRANSMISSION', 'TRANSMITTANCE']:
                sample['y'] = 1.0 - sample['y']
                sample['yunits'] = 'ABSORBANCE'

            # Нормализация X (1/CM)
            if 'xunits' in sample and str(sample['xunits']).upper() == 'MICROMETERS':
                sample['x'] = 10000.0 / sample['x']
                sample['xunits'] = '1/CM'

            # Разворот оси X при необходимости
            if sample['x'][0] > sample['x'][-1]:
                sample['x'] = np.flip(sample['x'])
                sample['y'] = np.flip(sample['y'])

            cas_number = str(sample.get('cas registry no', '')).strip() or os.path.splitext(os.path.basename(file))[0]
            row_data = {'CAS': cas_number}

            for key, value in sample.items():
                if key not in ['x', 'y']:
                    row_data[str(key)] = str(value)

            # RDKit расчеты структуры и свойств
            smiles_in = str(sample.get('smiles', '')).strip()
            inchi_in = str(sample.get('inchi', '')).strip()

            mol = None
            if smiles_in and smiles_in.lower() != 'nan':
                mol = Chem.MolFromSmiles(smiles_in)
            if mol is None and inchi_in and inchi_in.lower() != 'nan':
                mol = inchi.MolFromInchi(inchi_in)

            if mol is not None:
                try:
                    row_data['SMILES'] = Chem.MolToSmiles(mol, canonical=True)
                    row_data['InChi'] = inchi.MolToInchi(mol)
                    row_data['InChiKey'] = inchi.MolToInchiKey(mol)
                    row_data['Formula'] = rdMolDescriptors.CalcMolFormula(mol)
                    row_data['Mw'] = float(Descriptors.ExactMolWt(mol))
                except Exception:
                    pass

            # Интерполяция
            x_interp, y_interp = universal_interpolate(
                sample['x'], sample['y'], borders=borders, deltax=deltax
            )

            row_data['x1'] = x_interp.astype(np.float32)
            row_data['y1'] = y_interp.astype(np.float32)

            dataset_list.append(row_data)

        except Exception as e:
            if len(error_examples) < 3:
                error_examples.append(f"Файл {os.path.basename(file)}: {e}")
            continue

    print(f"\nОбработка завершена. Успешно: {len(dataset_list)}, Пропущено/Пустых: {empty_files_counter}")
    if error_examples:
        print("Примеры ошибок при обработке:")
        for err in error_examples:
            print(" -", err)

    # Формирование и очистка DataFrame
    df = pd.DataFrame(dataset_list)
    del dataset_list
    gc.collect()

    if not df.empty and 'y1' in df.columns:
        df = df.dropna(subset=['y1'])

    if not df.empty:
        rdkit_cols = ['SMILES', 'InChi', 'InChiKey', 'Mw', 'Formula']
        rename_dict = {
            col: col.replace('_', ' ').title() 
            for col in df.columns 
            if col not in ['x1', 'y1', 'CAS'] + rdkit_cols
        }
        df = df.rename(columns=rename_dict)

        if 'Formula' not in df.columns:
            df['Formula'] = np.nan

        if 'Molform' in df.columns:
            df['Formula'] = df['Formula'].fillna(df['Molform'])

        cols_to_drop = ['Molform', 'Cas Registry No', 'Cas_registry_no', 'Name', 'Data Type', 'Cas Name']
        df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

    # Сохранение результатов
    if save_path and not df.empty:
        dir_name = os.path.dirname(save_path)
        if dir_name:
            os.makedirs(dir_name, exist_ok=True)
        df.to_pickle(save_path)
        print(f'Файл успешно сохранен: {save_path}')

    return df

In [ ]:
def load_new_spectra(path_to_new_data: Union[str, Path]) -> pd.DataFrame:
    """Загружает и валидирует входные локальные данные."""
    if not path_to_new_data:
        print("Путь к данным не задан.")
        return pd.DataFrame()

    path_obj = Path(path_to_new_data)
    if not path_obj.exists() and not list(path_obj.parent.glob(path_obj.name)):
        print(f"Указанный путь не существует: {path_to_new_data}")
        return pd.DataFrame()

    print(f"Инициализация загрузки спектров: {path_to_new_data}")
    new_df = process_spectrum_folder(path_to_new_data)

    if new_df.empty:
        print("В указанной директории нет подходящих файлов данных.")

    return new_df

In [ ]:
rdBase.DisableLog('rdApp.*')
RDLogger.DisableLog('rdApp.*')

# Единая HTTP сессия
session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})

def fetch_smiles_smart(item):
    '''Каскадный поиск SMILES в онлайн-базах данных
    Приоритет поиска:
    PubChem API — поиск по нормализованному CAS.
    NCI Cactus API — резервный поиск по CAS (с фильтрацией HTML-ошибок).
    PubChem API — поиск по названию (title), если CAS отсутствует или не дал результатов.
    '''
    
    idx, cas, title = item
    norm_cas = normalize_cas(cas)
    
    # Попытка по CAS через PubChem xref/RN
    if norm_cas:
        try:
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/xref/RN/{norm_cas}/property/CanonicalSMILES/JSON"
            r = session.get(url, timeout=3)
            if r.status_code == 200:
                smiles = r.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
                return idx, smiles
        except Exception:
            pass

        # Резерв по CAS через NCI Cactus
        try:
            url = f"https://cactus.nci.nih.gov/chemical/structure/{norm_cas}/smiles"
            r = session.get(url, timeout=3)
            if r.status_code == 200 and "<html>" not in r.text and "Page not found" not in r.text:
                return idx, r.text.strip()
        except Exception:
            pass

    # Поиск по Title, если по CAS не удалось найти
    if title and str(title).lower() not in ['nan', 'none', '']:
        clean_title = str(title).strip()
        try:
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{clean_title}/property/CanonicalSMILES/JSON"
            r = session.get(url, timeout=3)
            if r.status_code == 200:
                smiles = r.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
                return idx, smiles
        except Exception:
            pass

    return idx, None


def fast_enrich_and_sync(df, db_path, max_threads=6):
    df = df.copy()

    '''
    Многопоточное обогащение датафрейма отсутствующими SMILES и автоматический расчет 
    химических метаданных через RDKit с синхронизацией в локальный .pkl 
    '''
    # Гарантируем наличие колонок
    for col in ['SMILES', 'InChI', 'InChIKey', 'Formula', 'Mw']:
        if col not in df.columns:
            df[col] = np.nan

    # Отбираем только те строки, где нет SMILES
    missing_mask = df['SMILES'].isna() | (df['SMILES'] == 'None') | (df['SMILES'] == 'Unknown') | (df['SMILES'] == '')
    items_to_fetch = [
        (idx, row.get('CAS'), row.get('Title')) 
        for idx, row in df[missing_mask].iterrows()
    ]

    print(f"Строк без SMILES для обработки: {len(items_to_fetch)}")

    smiles_dict = {}
    if items_to_fetch:
        with ThreadPoolExecutor(max_workers=max_threads) as executor:
            futures = [executor.submit(fetch_smiles_smart, item) for item in items_to_fetch]
            for future in tqdm(as_completed(futures), total=len(items_to_fetch), desc="Поиск SMILES (CAS + Title)"):
                idx, smiles = future.result()
                if smiles:
                    smiles_dict[idx] = smiles

        print(f"\n Успешно найдено SMILES: {len(smiles_dict)}")
        
        # Обновляем SMILES по индексам
        for idx, smiles in smiles_dict.items():
            df.loc[idx, 'SMILES'] = smiles

    # Расчет метаданных RDKit без вывода сообщений 
    print(" Расчет InChI, InChIKey, Mw и Formula в RDKit...")
    
    inchis, inchi_keys, formulas, mws = [], [], [], []

    for smiles in tqdm(df['SMILES'], desc="Локальная обработка RDKit"):
        smiles_str = str(smiles).strip() if pd.notna(smiles) and str(smiles).lower() not in ['unknown', 'none', 'nan', ''] else None
        
        mol = Chem.MolFromSmiles(smiles_str) if smiles_str else None
        if mol is not None:
            try:
                inchis.append(Chem.MolToInchi(mol))
                inchi_keys.append(Chem.MolToInchiKey(mol))
                formulas.append(rdMolDescriptors.CalcMolFormula(mol))
                mws.append(round(float(Descriptors.MolWt(mol)), 4))
                continue
            except Exception:
                pass

        inchis.append('None')
        inchi_keys.append('None')
        formulas.append('None')
        mws.append(np.nan)

    df['InChI'] = inchis
    df['InChIKey'] = inchi_keys
    df['Formula'] = formulas
    df['Mw'] = mws

    # Сохраняем результат
    df.to_pickle(db_path)
    print(f"\n База сохранена! Всего строк: {len(df)}")
    return df

In [ ]:
def deduplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Удаляет дубликаты столбцов (как с точными именами, так и с разным регистром, 
    например 'Inchikey' и 'InChIKey'). 
    
    При наличии нескольких одноименных колонок функция объединяет их данные 
    (выбирает непустые значения) и оставляет только один итоговый столбец.
    """
    df = df.copy()

    # Канонические имена для приведения регистра к единому стандарту
    canonical_map = {
        'cas': 'CAS',
        'title': 'Title',
        'smiles': 'SMILES',
        'inchi': 'InChI',
        'inchikey': 'InChIKey',
        'formula': 'Formula',
        'mw': 'Mw'
    }

    # Группируем все имеющиеся колонки по их нижнему регистру (lowercase)
    grouped_cols = {}
    for original_col in df.columns:
        low_col = str(original_col).strip().lower()
        target_name = canonical_map.get(low_col, original_col)
        grouped_cols.setdefault(target_name, []).append(original_col)

    cleaned_data = {}

    # Обрабатываем каждую группу колонок
    for target_col_name, orig_col_list in grouped_cols.items():
        if len(orig_col_list) == 1:
            # Если колонка уникальна, оставляем как есть
            cleaned_data[target_col_name] = df[orig_col_list[0]]
        else:
            # Если колонка встречается несколько раз:
            # Получаем все эти столбцы в виде одного под-датафрейма
            cols_subset = df.iloc[:, [i for i, col in enumerate(df.columns) if col in orig_col_list]]
            
            # Заменяем строковые 'None', 'Unknown', 'nan' и пустые строки на честный np.nan
            cols_clean = cols_subset.replace(['None', 'Unknown', 'nan', 'NaN', 'None', ''], np.nan)
            
            # Объединяем значения слева направо (первое непустое значение выигрывает)
            merged_series = cols_clean.bfill(axis=1).iloc[:, 0]
            
            cleaned_data[target_col_name] = merged_series

    result_df = pd.DataFrame(cleaned_data, index=df.index)
    
    removed_count = len(df.columns) - len(result_df.columns)
    print(f"Очистка завершена! Было колонок: {len(df.columns)}, стало: {len(result_df.columns)} (удалено дубликатов: {removed_count}).")
    
    return result_df

In [ ]:
COLUMN_MAPPING = {
    'cas': 'CAS',
    'cas_number': 'CAS',
    'title': 'Title',
    'name': 'Title',
    'jcamp_dx': 'Jcamp-Dx',
    'filepath': 'Filename',
    'smiles': 'SMILES',
    'inchi': 'InChI',
    'inchikey': 'InChIKey'
}
##Отвечает за управление базой данных, вызывает load_new_spectra внутри себя
def integrate_new_spectra(path_to_new_data: Union[str, Path], main_db_path: str,column_mapping: dict = COLUMN_MAPPING) -> pd.DataFrame:
    """
    Загружает новые спектры, нормализует названия колонок, 
    объединяет с основной базой и удаляет дубликаты.
    """
    # Загрузка новых данных
    new_df = load_new_spectra(path_to_new_data)
    if new_df.empty:
        print("Новые данные не загружены. Возвращаем текущую базу без изменений.")
        return pd.read_pickle(main_db_path) if os.path.exists(main_db_path) else pd.DataFrame()

    # Переименование колонок в новых данных (по словарю стандартизации)
    new_df = new_df.rename(columns=column_mapping)

    # Загрузка основной базы данных
    if os.path.exists(main_db_path):
        main_df = pd.read_pickle(main_db_path)
        print(f"Загружена основная база: {len(main_df)} строк.")
    else:
        print("Основная база не найдена. Будет создана новая.")
        main_df = pd.DataFrame()

    # Объединение строк (pd.concat сам соберет все колонки)
    combined_df = pd.concat([main_df, new_df], ignore_index=True)

    # Автоматическая очистка дубликатов колонок
    if 'deduplicate_columns' in globals():
        combined_df = deduplicate_columns(combined_df)

    # Удаление дубликатов строк по ключевым полям
    dedup_subset = [col for col in ['CAS', 'Filename', 'Title'] if col in combined_df.columns]
    if dedup_subset:
        initial_len = len(combined_df)
        combined_df = combined_df.drop_duplicates(subset=dedup_subset, keep='last').reset_index(drop=True)
        print(f"Удалено дубликатов строк: {initial_len - len(combined_df)}")

    # 7. Сохранение обновленной базы на диск
    combined_df.to_pickle(main_db_path)
    print(f" Успешно интегрировано! Итого строк в базе: {len(combined_df)}")

    return combined_df

In [ ]:
def select_spectral_metadata(df: pd.DataFrame, show: bool = True, n_rows: int = 10) -> pd.DataFrame:
    """
    Оставляет в датафрейме только ключевые метаданные спектров:
    CAS, Title, InChi, SMILES, Mw, State, Pressure, Temperature.
    
    Parameters:
    df : pd.DataFrame
        Исходный датафрейм со спектрами и метаданными.
    show : bool, default True
        Выводить ли отформатированную таблицу
    n_rows : int, default 10
        Количество строк для предпросмотра при show=True.
        
    Returns:
    pd.DataFrame
        Очищенный датафрейм со строго заданной структурой колонок.
    """
    target_columns = ['CAS', 'Title', 'InChi', 'SMILES', 'Mw', 'State', 'Pressure', 'Temperature', 'alkane', 'methyl', 'alkene', 'alkyne',
       'alcohols', 'amines', 'nitriles', 'aromatics', 'alkyl halides',
       'esters', 'ketones', 'aldehydes', 'carboxylic acids', 'ether',
       'acyl halides', 'amides', 'nitro']
    
    if df.empty:
        print("Передан пустой датафрейм.")
        return pd.DataFrame(columns=target_columns)

    df_clean = df.copy()

    # Приводим регистр колонок к совпадению с целевым списком (нечувствительность к регистру)
    col_mapping = {}
    target_lower = {col.lower(): col for col in target_columns}
    
    for col in df_clean.columns:
        col_lower = col.strip().lower()
        if col_lower in target_lower:
            col_mapping[col] = target_lower[col_lower]
            
    df_clean = df_clean.rename(columns=col_mapping)

    # Добавляем колонки из целевого списка, если их вовсе не было в исходных данных
    for col in target_columns:
        if col not in df_clean.columns:
            df_clean[col] = None

    # Фильтруем и упорядочиваем колонки
    result_df = df_clean[target_columns].copy()

    if show:
        print(f" Отображены метаданные для первых {min(n_rows, len(result_df))} записей (всего: {len(result_df)}):")
        # Использование display() дает красивую HTML-таблицу в Jupyter Notebook
        display(result_df.head(n_rows))

    return result_df

In [ ]:
def enrich_from_local_csv(df: pd.DataFrame, csv_path: str) -> pd.DataFrame:
    """
    Обогащает базу данных, моментально подтягивая SMILES, InChI и InChIKey 
    по CAS-номеру из локального файла compounds.csv.
    """
    df = df.copy()
    
    # Загружаем локальный справочник
    try:
        ref_df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Ошибка при чтении файла {csv_path}: {e}")
        return df

    # Гарантируем, что целевые колонки существуют
    for col in ['SMILES', 'InChI', 'InChIKey']:
        if col not in df.columns:
            df[col] = np.nan

    # Очищаем CAS от пробелов для идеального совпадения
    ref_df['cas'] = ref_df['cas'].astype(str).str.strip()
    df['CAS_clean'] = df['CAS'].astype(str).str.strip()

    # Создаем словари для поиска 
    smiles_map = ref_df.set_index('cas')['smiles'].dropna().to_dict()
    inchi_map = ref_df.set_index('cas')['inchi'].dropna().to_dict()
    inchikey_map = ref_df.set_index('cas')['inchikey'].dropna().to_dict()

    # Вспомогательная функция для поиска "пустых" ячеек
    def get_missing_mask(col_name):
        return df[col_name].isna() | df[col_name].isin(['None', 'Unknown', '', 'nan'])

    # Подтягиваем данные
    for target_col, data_map in zip(['SMILES', 'InChI', 'InChIKey'], [smiles_map, inchi_map, inchikey_map]):
        missing_mask = get_missing_mask(target_col)
        
        # Берем CAS из пустых строк, ищем их в словаре, и если находим — заполняем
        mapped_values = df.loc[missing_mask, 'CAS_clean'].map(data_map)
        df.loc[missing_mask, target_col] = np.where(
            mapped_values.notna(), 
            mapped_values, 
            df.loc[missing_mask, target_col]
        )

    # Убираем временную колонку очищенного CAS
    df = df.drop(columns=['CAS_clean'])

    # Подсчитываем результаты
    found_smiles = get_missing_mask('SMILES').sum()
    print(f" Поиск по локальному файлу завершен. Осталось пустых SMILES: {found_smiles}")
    
    return df

### Удаление дубликатов по CAS

In [ ]:
def remove_least_informative_duplicates(
    df: pd.DataFrame, main_db_path: str = None
) -> pd.DataFrame:
    """Дедупликация по CAS с отбором наиболее информативной строки."""
    df = df.copy()
    cas_series = df['CAS'].astype(str).str.strip().str.lower()
    valid_mask = df['CAS'].notna() & ~cas_series.isin(
        ['nan', 'none', 'unknown', '']
    )

    df_valid, df_invalid = df[valid_mask].copy(), df[~valid_mask].copy()
    if df_valid.empty:
        return df

    weights = {
        'SMILES': 10,
        'InChIKey': 10,
        'InChI': 8,
        'Mw': 5,
        'Formula': 5,
        'Title': 3,
        'y1': 5,
        'x1': 5,
    }

    def calc_score(row):
        '''Считает суммарный вес заполненных полей таблицы.
           Игнорирует NaN, 'none', 'unknown' и пустые списки/массивы'''
        score = 0
        for col, val in row.items():
            if val is None:
                continue
            if isinstance(val, (np.ndarray, list)):
                if len(val) > 0:
                    score += weights.get(col, 1)
            elif (
                pd.notna(val)
                and str(val).strip().lower() not in ['none', 'unknown', 'nan', '']
            ):
                score += weights.get(col, 1)
        return score

    df_valid['info_score'] = df_valid.apply(calc_score, axis=1)
    df_valid = df_valid.sort_values(
        by=['CAS', 'info_score'], ascending=[True, False]
    )
    df_dedup = df_valid.drop_duplicates(subset=['CAS'], keep='first').drop(
        columns=['info_score']
    )

    df_final = pd.concat([df_dedup, df_invalid], ignore_index=True)
    if main_db_path:
        df_final.to_pickle(main_db_path)
    return df_final



## Запуск

In [ ]:
def run_data_processing_pipeline(
    new_data_folder: str, db_path: str, csv_path: str
) -> pd.DataFrame:
    """Главный конвейер обработки файлов."""
    df = integrate_new_spectra(new_data_folder, db_path)
    df = deduplicate_columns(df)
    df = enrich_from_local_csv(df, csv_path)
    df = fast_enrich_and_sync(df, db_path)
    df = remove_least_informative_duplicates(df, main_db_path=db_path)
    print(f"Обработка завершена. Всего спектров в БД: {len(df)}")
    return df

In [ ]:
def run_full_program(
    new_data_folder: str,
    db_path: str,
    csv_path: str,
    force_relabel: bool = True
) -> pd.DataFrame:
    """
        Главная точка входа в программу.
        
        Автоматически определяет:
        - Создание БД с нуля (если db_path не существует).
        - Дополнение существующей БД новыми спектрами (если db_path найден).
        
        Args:
            new_data_folder: Папка с новыми JCAMP (.jdx / .dx) файлами.
            db_path: Путь к основному файлу базы данных (.pkl).
            csv_path: Путь к локальному справочнику (compounds.csv).
            force_relabel: Выполнять ли разметку 17 классов после сборки/обновления.
        """
    db_file = Path(db_path)
    db_file.parent.mkdir(parent=True, exist_ok=True)
    if not db_file.exists():
        print(f"База данных по пути {db_path} не найдена, создание новой базы")
    else:
        initial_db = pd.read_pickle(db_path)
        print(f"Дозаполнение существующей базы данных")
        print(f"Размер текущей базы данных: {len(initial_db)}")
        
    df_processed = run_data_processing_pipeline(
        new_data_folder=new_data_folder,
        db_path=db_path,
        csv_path=csv_path
    )
    if df_processed.empty:
        print("База данных пуста, проверьте путь к исходным файлам")
    if force_relabel:
        df_final = generate_functional_group_labels(df_processed, main_db_path=db_path)
    else:
        df_final = df_processed
    print(f" Итоговое количество уникальных спектров в БД: {len(df_final)}")
    return df_final



if __name__ == '__main__':
    df_db = run_full_program(
        new_data_folder=NEW_DATA_FOLDER,
        db_path=DB_PATH,
        csv_path=CSV_PATH,
        force_relabel=True
    )

## Разметка на 17 классов

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from rdkit import Chem, RDLogger

# Отключаем лишние предупреждения RDKit
RDLogger.DisableLog('rdApp.*')

In [ ]:
FUNC_GRP_SMARTS = {
    'alkane': '[CX4;H0,H1,H2,H4]',
    'methyl': '[CH3]',
    'alkene': '[CX3]=[CX3]',
    'alkyne': '[CX2]#C',
    'alcohols': '[#6][OX2H]',
    'amines': '[NX3;H2,H1;!$(NC=O)]',
    'nitriles': '[NX1]#[CX2]',
    'aromatics': 'a', 
    'alkyl halides': '[#6][F,Cl,Br,I]',
    'esters': '[#6][CX3](=O)[OX2H0][#6]',
    'ketones': '[#6][CX3](=O)[#6]',
    'aldehydes': '[CX3H1](=O)[#6]',
    'carboxylic acids': '[CX3](=O)[OX2H1]',
    'ether': '[OD2]([#6])[#6]',
    'acyl halides': '[CX3](=[OX1])[F,Cl,Br,I]',
    'amides': '[NX3][CX3](=[OX1])[#6]',
    'nitro': '[$([NX3+](=O)[O-]),$([NX3](=O)=O)]' 
}

# Предварительная компиляция SMARTS-шаблонов для высокой скорости
COMPILED_PATTERNS = {name: Chem.MolFromSmarts(smarts) for name, smarts in FUNC_GRP_SMARTS.items()}

In [ ]:
def get_mol_safe(smiles, inchi):
    """Безопасное создание Mol объекта из SMILES или InChI."""
    if pd.notna(smiles) and str(smiles).strip() not in ['None', 'Unknown', 'nan', '']:
        mol = Chem.MolFromSmiles(str(smiles).strip())
        if mol is not None:
            return mol
            
    if pd.notna(inchi) and str(inchi).strip() not in ['None', 'Unknown', 'nan', '']:
        try:
            mol = Chem.MolFromInchi(str(inchi).strip())
            if mol is not None:
                return mol
        except Exception:
            pass
            
    return None

In [ ]:
def generate_functional_group_labels(df: pd.DataFrame, main_db_path: str = None) -> pd.DataFrame:
    """
    Генерирует 17 бинарных меток (0 или 1) присутствия функциональных групп 
    и добавляет их в виде новых столбцов в датафрейм.
    """
    df = df.copy()
    group_names = list(COMPILED_PATTERNS.keys())
    
    # Резервируем матрицу под бинарные метки
    labels_matrix = np.zeros((len(df), len(group_names)), dtype=np.float32)
    valid_mol_count = 0

    print(f"Разметка {len(df)} спектров на 17 функциональных групп...")

    for idx, row in tqdm(enumerate(df.itertuples()), total=len(df), desc="Обработка молекул"):
        smiles = getattr(row, 'SMILES', None)
        inchi = getattr(row, 'InChI', None)
        
        mol = get_mol_safe(smiles, inchi)
        
        if mol is not None:
            valid_mol_count += 1
            for g_idx, (g_name, pattern) in enumerate(COMPILED_PATTERNS.items()):
                if pattern is not None and mol.HasSubstructMatch(pattern):
                    labels_matrix[idx, g_idx] = 1.0
        else:
            # Если структуру распознать не удалось, заменяем метки на NaN
            labels_matrix[idx, :] = np.nan

    # Записываем метки в датафрейм
    for g_idx, g_name in enumerate(group_names):
        df[g_name] = labels_matrix[:, g_idx]

    print(f"\n Разметка завершена! Успешно обработано молекул: {valid_mol_count} из {len(df)}")
    
    if main_db_path:
        df.to_pickle(main_db_path)
        print(f" Сохранено в: {main_db_path}")

    return df

In [ ]:
def analyze_class_statistics(df: pd.DataFrame):
    """Выводит статистику и строит график распределения 17 функциональных групп."""
    group_cols = list(COMPILED_PATTERNS.keys())
    
    # Фильтруем только размеченные записи
    df_labeled = df.dropna(subset=group_cols)
    total_labeled = len(df_labeled)
    
    # Подсчет количества единиц по каждому классу
    class_counts = df_labeled[group_cols].sum().astype(int)
    class_percentages = (class_counts / total_labeled * 100).round(2)
    
    stats_df = pd.DataFrame({
        'Класс (Функциональная группа)': class_counts.index,
        'Кол-во соединений': class_counts.values,
        'Процент от базы (%)': class_percentages.values
    }).sort_values(by='Кол-во соединений', ascending=False).reset_index(drop=True)

    print("ТАБЛИЦА РАСПРЕДЕЛЕНИЯ КЛАССОВ")
    print(stats_df.to_string(index=False))

    # Подсчет мульти-меток (сколько групп одновременно в одной молекуле)
    groups_per_mol = df_labeled[group_cols].sum(axis=1)
    
    # Визуализация
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # График 1: Распределение по классам
    sns.barplot(data=stats_df, x='Кол-во соединений', y='Класс (Функциональная группа)', hue='Класс (Функциональная группа)', ax=axes[0], palette='viridis')
    axes[0].set_title(f'Частота функциональных групп (Всего размечено: {total_labeled})', fontsize=12)
    axes[0].set_xlabel('Число спектров с данной группой')

    # График 2: Сколько функциональных групп содержит одна молекула
    sns.countplot(x=groups_per_mol.astype(int), hue=groups_per_mol, ax=axes[1], palette='magma')
    axes[1].set_title('Распределение числа функциональных групп на 1 молекулу', fontsize=12)
    axes[1].set_xlabel('Количество совпавших ФГ в молекуле')
    axes[1].set_ylabel('Число молекул')

    plt.tight_layout()
    plt.show()

    return stats_df

In [ ]:
DB_PATH = BASE_DIR / "db" / "processed_dataset2807.pkl"
df = pd.read_pickle(DB_PATH)

# 1. Генерируем 17 бинарных колонок
df_with_groups = generate_functional_group_labels(df, main_db_path=DB_PATH)

# 2. Выводим статистику и графики
stats = analyze_class_statistics(df_with_groups)

In [ ]:
select_spectral_metadata(df)

### Диагностика качества данных в столбце
Рассчитывает и выводит процент отсутствующих значений('none', 'unknown', 'nan', '') в консоль.

In [ ]:
# Если датафрейм еще не загружен:
# df = pd.read_pickle(DB_PATH)

if 'InChI' in df.columns:
    # Приводим значения к строке в нижнем регистре для проверки на текстовые пустышки
    inchi_clean = df['InChI'].astype(str).str.strip().str.lower()
    
    # Ищем системные NaN или строки 'none', 'unknown', 'nan', ''
    missing_mask = df['InChI'].isna() | inchi_clean.isin(['none', 'unknown', 'nan', ''])
    
    missing_count = missing_mask.sum()
    total_count = len(df)
    percent = (missing_count / total_count * 100) if total_count > 0 else 0

    print(f"Записей без InChI: {missing_count} из {total_count} ({percent:.2f}%)")
else:
    print("Столбец 'InChI' не найден в датафрейме.")

## Визуализация молекул Rdkit
Выводит случайно выбранные молекулы из датафрейма в виде галереи с подписями (CAS и Название)

In [ ]:
def show_random_molecules_grid(df: pd.DataFrame, n_samples: int = 12, mols_per_row: int = 4):
    """Отображает сетку из случайных молекул датасета прямо в ячейке Jupyter Notebook."""
    # Фильтруем только записи со SMILES
    valid_df = df.dropna(subset=['SMILES'])
    if valid_df.empty:
        print("В датафрейме нет доступных SMILES для визуализации.")
        return None

    sample_df = valid_df.sample(min(n_samples, len(valid_df)))
    
    mols = []
    legends = []
    
    for _, row in sample_df.iterrows():
        mol = get_mol_safe(row.get('SMILES'), row.get('InChI'))
        if mol is not None:
            mols.append(mol)
            cas = str(row.get('CAS', 'No CAS'))
            title = str(row.get('Title', ''))[:18]
            legends.append(f"{cas}\n{title}")

    return Draw.MolsToGridImage(
        mols, 
        molsPerRow=mols_per_row, 
        subImgSize=(250, 250), 
        legends=legends
    )


In [ ]:
df_final = pd.read_pickle(DB_PATH)

In [ ]:
show_random_molecules_grid(df_final, n_samples=8, mols_per_row=4)